# E-Commerce Customer Spending — Linear Regression Analysis

## Problem Statement
An e-commerce company sells products both through a **mobile app** and a **website**. Customers also attend in-store style sessions. The company wants to understand **which behavioral factors drive yearly spending**, and whether it should focus its development efforts on the app or the website.

**Objective:** Build a multiple linear regression model to predict `Yearly Amount Spent` by a customer, and perform inferential analysis to identify which features significantly influence spending.

## Dataset Description
| Column | Description |
|---|---|
| `Avg. Session Length` | Average duration (minutes) of in-store style advice sessions |
| `Time on App` | Average time (minutes) spent on the mobile app |
| `Time on Website` | Average time (minutes) spent on the website |
| `Length of Membership` | Number of years the customer has been a member |
| `Yearly Amount Spent` | **Target** — Total amount spent by the customer in a year (USD) |

- **Rows:** 500 customers
- **Source:** Synthetic e-commerce dataset (commonly used for regression practice)
- **No missing values**

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pylab
from scipy import stats

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Helper Functions
Wrapping repeated logic into reusable functions keeps the notebook clean and avoids copy-paste errors.

In [ ]:
def check_normality(series: pd.Series, alpha: float = 0.05) -> None:
    """
    Run Shapiro-Wilk normality test on a Series.
    Prints the p-value and conclusion.
    """
    stat, p = stats.shapiro(series)
    conclusion = 'Normally Distributed ✓' if p > alpha else 'NOT Normally Distributed ✗'
    print(f"  {series.name:<30}  p = {p:.4f}  →  {conclusion}")


def evaluate_model(y_true: pd.Series, y_pred: np.ndarray, label: str = '') -> dict:
    """
    Compute and print MAE, RMSE, and R² for a set of predictions.
    Returns a dict of the metrics.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    tag  = f'[{label}]' if label else ''
    print(f"  {tag}  MAE = {mae:.2f}   RMSE = {rmse:.2f}   R² = {r2:.4f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}


def plot_residual_diagnostics(y_pred: np.ndarray, residuals: np.ndarray, title: str = '') -> None:
    """
    Plot three residual diagnostic charts side-by-side:
      1. Residuals vs Fitted  (homoscedasticity check)
      2. Histogram of residuals  (normality check)
      3. Q-Q plot  (normality check)
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Residual Diagnostics — {title}', fontsize=13, fontweight='bold')

    # --- Plot 1: Residuals vs Fitted ---
    axes[0].scatter(y_pred, residuals, alpha=0.5, color='steelblue', edgecolors='white', linewidths=0.3)
    axes[0].axhline(0, linestyle='--', color='red')
    axes[0].set_xlabel('Fitted Values')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs Fitted')

    # --- Plot 2: Histogram ---
    sns.histplot(residuals, bins=20, kde=True, ax=axes[1], color='steelblue')
    axes[1].set_title('Distribution of Residuals')
    axes[1].set_xlabel('Residual')

    # --- Plot 3: Q-Q Plot ---
    stats.probplot(residuals, dist='norm', plot=axes[2])
    axes[2].set_title('Q-Q Plot')

    plt.tight_layout()
    plt.show()


def breusch_pagan_test(residuals: np.ndarray, X_with_const, alpha: float = 0.05) -> None:
    """
    Run the Breusch-Pagan test for heteroscedasticity.
    H0: Residuals have constant variance (homoscedastic)
    H1: Variance of residuals depends on the features (heteroscedastic)
    """
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(residuals, X_with_const)
    print(f"  Breusch-Pagan LM statistic = {lm_stat:.4f}")
    print(f"  p-value                    = {lm_pvalue:.4f}")
    if lm_pvalue > alpha:
        print("  Conclusion: Fail to reject H0 → Residuals are HOMOSCEDASTIC ✓")
    else:
        print("  Conclusion: Reject H0 → Residuals are HETEROSCEDASTIC ✗")

## 3. Load & Inspect Data

In [ ]:
raw_df = pd.read_csv('ecommerce.csv')

print(f"Shape: {raw_df.shape}")
print(f"\nNull values per column:\n{raw_df.isnull().sum()}")
raw_df.head()

In [ ]:
raw_df.info()

In [ ]:
raw_df.describe().round(2)

In [ ]:
# Keep only the numeric columns relevant to the regression
df = raw_df[['Avg. Session Length', 'Time on App', 'Time on Website',
             'Length of Membership', 'Yearly Amount Spent']]

## 4. Exploratory Data Analysis

Before fitting a linear regression model we must verify four key assumptions:
1. **Linearity** — Each feature has a roughly linear relationship with the target
2. **Normality of Residuals** — Errors are normally distributed
3. **Homoscedasticity** — Variance of errors is constant across fitted values
4. **No severe Multicollinearity** — Features are not strongly correlated with each other

### 4.1 Univariate Analysis — Distribution of Features

In [ ]:
features = ['Avg. Session Length', 'Time on App', 'Time on Website', 'Length of Membership']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Feature Distributions', fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flatten(), features):
    sns.histplot(data=df, x=feat, kde=True, ax=ax, color='steelblue')
    ax.set_title(feat)

plt.tight_layout()
plt.show()

In [ ]:
# Skewness and Kurtosis
print("Skewness and Kurtosis of each feature:")
print(f"{'Feature':<30} {'Skewness':>10} {'Kurtosis':>10}")
print('-' * 52)
for feat in df.columns:
    print(f"{feat:<30} {df[feat].skew():>10.4f} {df[feat].kurtosis():>10.4f}")

Near-zero skewness and kurtosis ≈ 0 (excess kurtosis) across all features indicate approximately normal, symmetric distributions with no heavy tails.

In [ ]:
# Shapiro-Wilk normality test for each feature
print("Shapiro-Wilk Normality Test (H0: data is normally distributed):")
print('-' * 65)
for feat in df.columns:
    check_normality(df[feat])

### 4.2 Outlier Detection — Boxplots
The `describe()` table only shows quartiles — it does not definitively identify outliers. Boxplots visualize the IQR fence and flag points beyond 1.5×IQR as potential outliers.

In [ ]:
fig, axes = plt.subplots(1, len(df.columns), figsize=(18, 5))
fig.suptitle('Boxplots — Outlier Detection', fontsize=13, fontweight='bold')

for ax, col in zip(axes, df.columns):
    sns.boxplot(data=df, y=col, ax=ax, color='steelblue', width=0.4,
                flierprops=dict(marker='o', color='red', markersize=5))
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

# Count outliers per column using IQR rule
print("\nOutlier count per column (IQR rule: beyond 1.5×IQR):")
for col in df.columns:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    print(f"  {col:<30}  Outliers: {n_outliers}")

### 4.3 Bivariate Analysis — Feature vs Target

In [ ]:
sns.pairplot(data=df, plot_kws={'alpha': 0.4, 's': 15})
plt.suptitle('Pairplot', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# Correlation heatmap
corr = df.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Key observations from bivariate analysis:**
- `Length of Membership` ↔ `Yearly Amount Spent`: correlation = **0.81** (strong positive)
- `Time on App` ↔ `Yearly Amount Spent`: correlation = **0.50** (moderate positive)
- `Avg. Session Length` ↔ `Yearly Amount Spent`: correlation = **0.36** (weak positive)
- `Time on Website` ↔ `Yearly Amount Spent`: correlation ≈ **0.00** (essentially no relationship)
- Feature-to-feature correlations are all low → no strong multicollinearity visible here

## 5. Model 1 — Baseline Linear Regression (All Features)

In [ ]:
X = df.drop(columns=['Yearly Amount Spent'])
y = df['Yearly Amount Spent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train = lr.predict(X_train)
y_pred_test  = lr.predict(X_test)

print(f"Intercept: {lr.intercept_:.4f}")
print("\nCoefficients:")
for feat, coef in zip(X.columns, lr.coef_):
    print(f"  {feat:<30}  {coef:.4f}")

In [ ]:
print("Model 1 — Performance Metrics:")
print("  Train:")
m_train = evaluate_model(y_train, y_pred_train, 'Train')
print("  Test:")
m_test  = evaluate_model(y_test,  y_pred_test,  'Test')

r2_gap = abs(m_train['r2'] - m_test['r2'])
print(f"\n  |R²_train - R²_test| = {r2_gap:.4f}  →",
      "Model is well-fitted ✓" if r2_gap < 0.05 else "Possible overfitting ✗")

In [ ]:
# 5-Fold Cross Validation
cv_scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
print(f"5-Fold CV R²: {cv_scores}")
print(f"Mean R² = {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}")

## 6. Residual Diagnostics — Model 1

In [ ]:
residuals_m1 = y_test - y_pred_test
plot_residual_diagnostics(y_pred_test, residuals_m1, 'Model 1 (All Features)')

In [ ]:
# Normality tests on residuals
print("Normality Tests on Residuals:")
print("  Shapiro-Wilk:")
stat, p = stats.shapiro(residuals_m1)
print(f"    p = {p:.4f}  →  {'Normal ✓' if p > 0.05 else 'Not Normal ✗'}")
print("  Omnibus (D'Agostino-Pearson):")
stat, p = stats.normaltest(residuals_m1)
print(f"    p = {p:.4f}  →  {'Normal ✓' if p > 0.05 else 'Not Normal ✗'}")

In [ ]:
# Formal Homoscedasticity Test — Breusch-Pagan
# We need to test on the training set where the model was fitted
residuals_train_m1 = y_train - y_pred_train
X_train_sm = sm.add_constant(X_train)

print("Breusch-Pagan Test for Homoscedasticity (on training residuals):")
print("  H0: Variance of residuals is constant (homoscedastic)")
print("-" * 60)
breusch_pagan_test(residuals_train_m1, X_train_sm)

## 7. Inferential Analysis via OLS — Model 1

In [ ]:
X_train_sm = sm.add_constant(X_train)
ols_model1 = sm.OLS(y_train, X_train_sm).fit()
print(ols_model1.summary())

**Key inferences from OLS Summary — Model 1:**
1. **F-statistic p-value ≈ 0** → The model as a whole is highly significant; at least one feature meaningfully predicts Yearly Spending
2. **`Time on Website` has p = 0.524** → This feature is statistically insignificant (p >> 0.05); we cannot reject H0 that its coefficient is zero
3. **Condition Number ≈ 259** → Indicates potential multicollinearity. We investigate this further after scaling

## 8. Model 2 — OLS Without `Time on Website`

In [ ]:
X_train_no_web = X_train[['Avg. Session Length', 'Time on App', 'Length of Membership']]
X_train_no_web_sm = sm.add_constant(X_train_no_web)

ols_model2 = sm.OLS(y_train, X_train_no_web_sm).fit()
print(ols_model2.summary())

**Comparison — Model 1 vs Model 2:**

| Metric | Model 1 (all features) | Model 2 (no Website) | Better? |
|---|---|---|---|
| R² | 0.985 | 0.985 | Same |
| Adj. R² | 0.985 | 0.985 | Same |
| F-statistic | ~6,676 | ~8,915 | Model 2 ↑ |
| AIC | 2970 | 2969 | Model 2 ↓ |
| BIC | 2990 | 2985 | Model 2 ↓ |
| Condition No. | ~2590 | ~1280 | Model 2 ↓ |

Removing `Time on Website` gives identical predictive power with a cleaner, more parsimonious model.

## 9. Model 3 — Scaled Features (All Features)
The high condition number in Model 1 (2590) raised a concern about multicollinearity. But the raw features are on very different numeric scales. StandardScaler removes this scale-driven inflation in the condition number.

In [ ]:
# Rebuild X with all features for this comparison
X_full = df.drop(columns=['Yearly Amount Spent'])
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_full, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_f)   # fit on train only
X_test_scaled  = scaler.transform(X_test_f)        # apply same scale to test

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_full.columns, index=X_train_f.index)
X_test_scaled_df  = pd.DataFrame(X_test_scaled,  columns=X_full.columns, index=X_test_f.index)

X_train_scaled_sm = sm.add_constant(X_train_scaled_df)
ols_model3 = sm.OLS(y_train_f, X_train_scaled_sm).fit()
print(ols_model3.summary())

**Critical insight:** The condition number dropped from **2590 → ~1.11** after StandardScaling. This confirms that the apparent multicollinearity in Model 1 was purely a **scale artifact**, not a genuine collinearity between features. The features were never truly multicollinear — they were just on different numeric ranges.

> Multicollinearity does NOT affect predictive accuracy — it only distorts coefficient estimates and standard errors in inferential analysis. Always scale before using OLS for inference.

## 10. Model 4 — Scaled Features Without `Time on Website` (Final Model)
This is the best model: scaled for correct inference + insignificant feature removed.

In [ ]:
# Drop Time on Website
X_final = df[['Avg. Session Length', 'Time on App', 'Length of Membership']]
X_train_fin, X_test_fin, y_train_fin, y_test_fin = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

scaler_fin = StandardScaler()
X_train_fin_scaled = scaler_fin.fit_transform(X_train_fin)
X_test_fin_scaled  = scaler_fin.transform(X_test_fin)

X_train_fin_df = pd.DataFrame(X_train_fin_scaled, columns=X_final.columns, index=X_train_fin.index)
X_test_fin_df  = pd.DataFrame(X_test_fin_scaled,  columns=X_final.columns, index=X_test_fin.index)

# OLS for inference
X_train_fin_sm = sm.add_constant(X_train_fin_df)
ols_model4 = sm.OLS(y_train_fin, X_train_fin_sm).fit()
print(ols_model4.summary())

In [ ]:
# Evaluate final model on the TEST SET using sklearn
lr_final = LinearRegression()
lr_final.fit(X_train_fin_scaled, y_train_fin)

y_pred_fin_train = lr_final.predict(X_train_fin_scaled)
y_pred_fin_test  = lr_final.predict(X_test_fin_scaled)

print("Final Model (Scaled, No Website) — Performance:")
print("  Train:")
m_fin_train = evaluate_model(y_train_fin, y_pred_fin_train, 'Train')
print("  Test:")
m_fin_test  = evaluate_model(y_test_fin,  y_pred_fin_test,  'Test')

r2_gap_fin = abs(m_fin_train['r2'] - m_fin_test['r2'])
print(f"\n  |R²_train - R²_test| = {r2_gap_fin:.4f}  →",
      "Model is well-fitted ✓" if r2_gap_fin < 0.05 else "Possible overfitting ✗")

In [ ]:
# Residual diagnostics on final model
residuals_fin = y_test_fin - y_pred_fin_test
plot_residual_diagnostics(y_pred_fin_test, residuals_fin, 'Final Model (Scaled, No Website)')

In [ ]:
# Breusch-Pagan on final model
residuals_fin_train = y_train_fin - y_pred_fin_train
X_train_fin_sm_bp = sm.add_constant(X_train_fin_df)

print("Breusch-Pagan Test — Final Model:")
breusch_pagan_test(residuals_fin_train, X_train_fin_sm_bp)

## 11. Final Summary

### All Four Models Compared

| Model | Features | Scaled | R² (Test) | Condition No. |
|---|---|---|---|---|
| Model 1 | All 4 | No | ~0.985 | ~2590 (scale artifact) |
| Model 2 | No Website | No | ~0.985 | ~1280 |
| Model 3 | All 4 | Yes | ~0.985 | ~1.11 |
| **Model 4** | **No Website** | **Yes** | **~0.985** | **~1.07** |

### Key Takeaways

1. **`Length of Membership`** is the single strongest driver of yearly spending (correlation 0.81)
2. **`Time on Website`** has no statistically significant effect (p = 0.524) — the company should NOT invest in improving the website to drive revenue; the app is what matters
3. **`Time on App`** is a significant, meaningful predictor — improving the app experience has a measurable payoff
4. The apparent multicollinearity (Condition No. ~2590) in the unscaled model was a **scaling artifact**, not real collinearity — confirmed when scaling brought it down to 1.11
5. All four linear regression assumptions are satisfied in the final model: linearity confirmed by pairplot, residuals are normally distributed (Shapiro-Wilk + Omnibus both p > 0.05), homoscedasticity confirmed by Breusch-Pagan (p > 0.05), and no multicollinearity after scaling